# CIFAR-10 Image Classification using SIB-Net in PyTorch


# Necessary Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import CIFAR10
from torchvision.utils import make_grid
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import random_split, ConcatDataset
import torchvision.transforms as tt


____
#### Before we load the data, it is required to first prepare the transformations to be applied. It is an important step to prepare the data for training to avoid overfitting problem.
____

In [ ]:
# Per-channel (R, G, B) mean and std of the CIFAR-10 training set
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_transform = tt.Compose([
    tt.RandomCrop(32, padding=4),
    tt.RandomHorizontalFlip(),
    tt.ToTensor(),
    tt.Normalize(CIFAR_MEAN, CIFAR_STD)
])

test_transform = tt.Compose([
    tt.ToTensor(),
    tt.Normalize(CIFAR_MEAN, CIFAR_STD)
])

def denormalize(img):
    """Undo Normalize() so a 3x32x32 tensor can be displayed."""
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1).to(img.device)
    std  = torch.tensor(CIFAR_STD).view(3, 1, 1).to(img.device)
    return (img * std + mean).clamp(0, 1)

___
RandomHorizontalFlip randomly flips an image with a probability of 50%, and RandomCrop pads an image by 4 pixels on each side and then randomly crops 32x32 from the image after padding. We add such transformations to add noise to the data and prevent our model from overfitting. There are also other transformations you can use such as ColorJitter and RandomVerticalFlip etc. but these are sufficient for our purposes. Unlike digits, natural objects (cars, birds, horses, ...) keep their label when mirrored, so horizontal flipping is a safe augmentation for CIFAR-10. <br />

ToTensor simply converts the image to a Tensor. Since it's a coloured image, it has 3 channels (R,G,B) so the Tensor is of size 3x32x32. <br/>

Normalize takes the mean and standard deviation for each channel of the entire dataset as input (0.4914, 0.4822, 0.4465 and 0.2470, 0.2435, 0.2616 for CIFAR-10). Normalizing scales our data to a similar range of values to make sure that our gradients don't go out of control.
Now we just prepare our train and test dataset and then we can explore the data.
___

# Loading Data

In [ ]:
from torchvision import datasets, transforms

# Load CIFAR-10
train_data = datasets.CIFAR10(root="./data", train=True, transform=train_transform, download=True)
test_data  = datasets.CIFAR10(root="./data", train=False, transform=test_transform, download=True)


_____

In [ ]:
for image, label in train_data:
    print("Image shape: ",image.shape)
    print("Image tensor: ", image)
    print("Label: ", label)
    break

In [ ]:
train_classes_items = dict()

for train_item in train_data:
    label = train_data.classes[train_item[1]]
    if label not in train_classes_items:
        train_classes_items[label] = 1
    else:
        train_classes_items[label] += 1

train_classes_items

In [ ]:
test_classes_items = dict()
for test_item in test_data:
    label = test_data.classes[test_item[1]]
    if label not in test_classes_items:
        test_classes_items[label] = 1
    else:
        test_classes_items[label] += 1

test_classes_items

# BATCHSIZE & DataLoader

In [ ]:
BATCH_SIZE = 64
train_dl = DataLoader(train_data, BATCH_SIZE, num_workers=4, pin_memory=True, shuffle=True)
test_dl = DataLoader(test_data, BATCH_SIZE, num_workers=4, pin_memory=True)

# Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torchvision

train_8_samples = DataLoader(train_data, 8, num_workers=4, pin_memory=True, shuffle=True)

def imshow(img):
    img = denormalize(img)       # unnormalize (CIFAR-10 stats)
    npimg = img.numpy()
    plt.figure(figsize=(10, 10))
    if npimg.shape[0] == 1:  # grayscale
        plt.imshow(npimg[0], cmap="gray")
    else:  # RGB
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

dataiter = iter(train_8_samples)
images, labels = next(dataiter)

imshow(torchvision.utils.make_grid(images))
print(' '.join(f'{train_data.classes[labels[j]]}' for j in range(8)))


# Get CUDA ready

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def to_device(data,device):
    if isinstance(data,(list,tuple)):
        return [to_device(x,device) for x in data]
    return data.to(device,non_blocking=True)


class ToDeviceLoader:
    def __init__(self,data,device):
        self.data = data
        self.device = device
        
    def __iter__(self):
        for batch in self.data:
            yield to_device(batch,self.device)
            
    def __len__(self):
        return len(self.data)


In [ ]:
device = get_device()
print(device)

train_dl = ToDeviceLoader(train_dl, device)
test_dl = ToDeviceLoader(test_dl, device)

In [ ]:
def accuracy(predicted, actual):
    _, predictions = torch.max(predicted, dim=1)
    return torch.tensor(torch.sum(predictions==actual).item()/len(predictions))

In [ ]:
import torch.nn.utils.prune as prune

def apply_pruning(model, amount=0.2):
    """
    Apply unstructured L1 pruning to all linear and conv layers.
    'amount' = fraction of weights to prune in each layer
    """
    for module in model.modules():
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            prune.l1_unstructured(module, name="weight", amount=amount)


# SIB-Net Implementation

BaseModel allows us to check and record the results of our model every time we train and pretty much just helps us keep track of our progress. This is what our actual model will inherit. <br />

The proposed model is the **full model** from the ablation study (`basconv-mbinception.ipynb`): Inception-style multi-branch blocks built on **Blueprint Separable Convolutions (BSConv-U)**, wrapped in residual blocks with linearly-scaled stochastic depth (`drop_path_rate=0.1`). Squeeze-and-Excitation is no longer part of the architecture. Adapted here for CIFAR-10: `in_ch=3`, `num_classes=10`, 32x32 RGB input.<br />
____


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# BaseModel with optional L1
# -----------------------------
class BaseModel(nn.Module):
    def __init__(self):
        super().__init__()

    def training_step(self, batch, l1_lambda: float = 0.0):
        images, targets = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, targets)
        if l1_lambda > 0:
            l1_norm = sum(p.abs().sum() for p in self.parameters())
            loss = loss + l1_lambda * l1_norm
        return loss

    def validation_step(self, batch):
        images, targets = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, targets)
        acc = (outputs.argmax(dim=1) == targets).float().mean()
        return {"val_loss": loss.detach(), "val_acc": acc}

    def validation_epoch_end(self, outputs):
        batch_losses = [x["val_loss"] for x in outputs]
        batch_accs = [x["val_acc"] for x in outputs]
        return {
            "val_loss": torch.stack(batch_losses).mean().item(),
            "val_acc": torch.stack(batch_accs).mean().item(),
        }

# -----------------------------
# DropPath (Stochastic Depth)
# -----------------------------
class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep_prob
        return x.div(keep_prob) * mask

# -----------------------------
# Blueprint Separable Convolution (BSConv-U)
# -----------------------------
class BSConv2d(nn.Module):
    """Blueprint Separable Convolution -- unconstrained variant (Haase &
    Amthor, "Rethinking Depthwise Separable Convolutions: How Intra-Kernel
    Correlations Lead to Improved MobileNets", CVPR 2020).

    Standard depthwise-separable conv applies a depthwise (per-channel,
    spatial-only) conv first and mixes channels with a 1x1 pointwise conv
    second. BSConv reverses that order: a full-rank 1x1 pointwise conv mixes
    channels *first*, producing out_ch "blueprint" feature maps, and a cheap
    depthwise conv then applies a single learned k x k spatial filter to each
    of those output channels.

    One internal BN+ReLU after the pointwise conv (so the pair isn't two
    back-to-back linear ops), and a final linear (no BN/ReLU) depthwise stage,
    since the outer block owns all block-level BN/activation handling.
    """

    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0):
        super().__init__()
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        # Depthwise: one learned k x k spatial filter per output channel --
        # the "blueprint" for that channel, applied after channels are
        # already mixed by the pointwise conv above.
        self.depthwise = nn.Conv2d(
            out_ch, out_ch, kernel_size, stride, padding, groups=out_ch, bias=False
        )

    def forward(self, x):
        x = self.pointwise(x)
        x = self.depthwise(x)
        return x


class RegularConv2d(nn.Module):
    """Standard (non-BSConv) convolution, same interface as BSConv2d."""

    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, bias=False)

    def forward(self, x):
        return self.conv(x)


def make_conv(in_ch, out_ch, kernel_size, stride, padding, bsconv):
    if bsconv:
        return BSConv2d(in_ch, out_ch, kernel_size, stride, padding)
    return RegularConv2d(in_ch, out_ch, kernel_size, stride, padding)

# -----------------------------
# Inception Block (lightweight, BSConv branches)
# supports a stride so the block can downsample when needed
# -----------------------------
class InceptionBlockLight(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, use_bsconv=True):
        super().__init__()
        ch_1x1    = out_ch // 4
        remaining = out_ch - ch_1x1
        ch_3x3    = remaining // 3
        ch_5x5    = remaining // 3
        ch_pool   = out_ch - (ch_1x1 + ch_3x3 + ch_5x5)
        assert ch_pool > 0, f"channel split invalid: ch_pool={ch_pool} for out_ch={out_ch}"

        self.path1 = nn.Sequential(
            nn.Conv2d(in_ch, ch_1x1, 1, stride, 0, bias=False),
            nn.BatchNorm2d(ch_1x1),
            nn.ReLU(inplace=True),
        )

        self.path2 = nn.Sequential(
            make_conv(in_ch, ch_3x3, 3, stride, 1, bsconv=use_bsconv),
            nn.BatchNorm2d(ch_3x3), nn.ReLU(inplace=True),
        )
        self.path3 = nn.Sequential(
            make_conv(in_ch, ch_5x5, 5, stride, 2, bsconv=use_bsconv),
            nn.BatchNorm2d(ch_5x5), nn.ReLU(inplace=True),
        )

        self.path4 = nn.Sequential(
            nn.MaxPool2d(3, stride, 1),
            nn.Conv2d(in_ch, ch_pool, 1, bias=False),
            nn.BatchNorm2d(ch_pool), nn.ReLU(inplace=True),
        )

        self.project = nn.Conv2d(out_ch, out_ch, 1, bias=False)

    def forward(self, x):
        out = torch.cat(
            [self.path1(x), self.path2(x), self.path3(x), self.path4(x)], dim=1
        )
        return self.project(out)


class PlainConvBlock(nn.Module):
    """Single-branch replacement for InceptionBlockLight (no-inception variant).
    use_bsconv stays orthogonal so the same flag still applies."""

    def __init__(self, in_ch, out_ch, stride=1, use_bsconv=True):
        super().__init__()
        self.conv = make_conv(in_ch, out_ch, 3, stride, 1, bsconv=use_bsconv)

    def forward(self, x):
        return self.conv(x)

# -----------------------------
# Residual block wrapping two feature blocks
# -----------------------------
class ResNetBlock(nn.Module):
    def __init__(
        self,
        in_ch,
        out_ch,
        stride=1,
        drop_path_prob=0.0,
        use_inception=True,
        use_bsconv=True,
        use_skip=True,
    ):
        super().__init__()
        self.use_skip = use_skip
        block_kwargs = dict(use_bsconv=use_bsconv)
        block_cls = InceptionBlockLight if use_inception else PlainConvBlock

        self.feat1 = block_cls(in_ch, out_ch, stride, **block_kwargs)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.feat2 = block_cls(out_ch, out_ch, 1, **block_kwargs)
        self.bn2 = nn.BatchNorm2d(out_ch)

        if use_skip:
            if in_ch != out_ch or stride != 1:
                self.adjust = nn.Sequential(
                    nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                    nn.BatchNorm2d(out_ch),
                )
            else:
                self.adjust = nn.Identity()
        else:
            self.adjust = None

        self.drop_path = DropPath(drop_path_prob)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.feat1(x)
        out = self.act(self.bn1(out))
        out = self.feat2(out)
        out = self.bn2(out)
        out = self.drop_path(out)
        if self.use_skip:
            out = out + self.adjust(x)
        return self.act(out)

# -----------------------------
# Full SIB-Net model (BSConv + Inception + residual)
# -----------------------------
class SIBNet(BaseModel):
    """Full model from the ablation study: Inception-style multi-branch blocks
    with Blueprint Separable Convolutions and residual connections, plus
    linearly-scaled stochastic depth. Squeeze-and-Excitation is not used.

    Defaults are set for CIFAR-10: 3 input channels, 10 classes, 32x32.
    """

    def __init__(
        self,
        in_ch=3,
        num_classes=10,
        base_filters=64,
        num_blocks=(2, 2, 2),
        drop_path_rate=0.1,
        use_inception=True,
        use_bsconv=True,
        use_skip=True,
    ):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, base_filters, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_filters),
            nn.ReLU(inplace=True),
        )

        stages = []
        channels = base_filters
        total_blocks = sum(num_blocks)
        block_idx = 0
        for stage_idx, n_blocks in enumerate(num_blocks):
            out_ch = base_filters * (2 ** stage_idx)
            for b in range(n_blocks):
                stride = 2 if (stage_idx > 0 and b == 0) else 1
                this_drop = drop_path_rate * (block_idx / max(1, total_blocks - 1))
                stages.append(
                    ResNetBlock(
                        channels,
                        out_ch,
                        stride,
                        this_drop,
                        use_inception=use_inception,
                        use_bsconv=use_bsconv,
                        use_skip=use_skip,
                    )
                )
                channels = out_ch
                block_idx += 1

        self.stages = nn.Sequential(*stages)
        self.conv3x3 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(channels, num_classes)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        out = self.stem(x)
        out = self.stages(out)
        out = self.conv3x3(out)
        out = self.pool(out).view(out.size(0), -1)
        out = self.dropout(out)
        out = self.fc(out)
        return out

    def summary(self, input_shape=(1, 3, 32, 32)):
        """Pretty-print a summary using torchsummary (optional dependency)."""
        try:
            from torchsummary import summary
            summary(self, input_shape[1:])
        except Exception as e:
            print("Install torchsummary to use model.summary(). Error:", e)


# Backwards-compatible alias, so any cell still referring to the old class name works.
ResNetInceptionSE = SIBNet

# -----------------------------
# Build the model used by the rest of the notebook
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SIBNet(
    in_ch=3,
    num_classes=10,
    base_filters=64,
    num_blocks=(2, 2, 2),
    drop_path_rate=0.1,
    use_inception=True,
    use_bsconv=True,
    use_skip=True,
).to(device)

# sanity check on a CIFAR-10-shaped batch
_x = torch.randn(2, 3, 32, 32, device=device)
print("Output shape:", model(_x).shape)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


# Evaluation & Fit Function for Training

In [ ]:
@torch.no_grad()
def evaluate(model,test_dl):
    model.eval()
    outputs = [model.validation_step(batch) for batch in test_dl]
    return model.validation_epoch_end(outputs)

In [ ]:
def fit(model, train_dl, val_dl, optimizer, epochs, grad_clip=None, scheduler=None):
    """
    Train the model (OneCycleLR-compatible version).
    """
    history = []

    for epoch in range(epochs):
        model.train()
        train_losses, train_accs = [], []

        for images, targets in train_dl:
            images, targets = images.to(next(model.parameters()).device), targets.to(next(model.parameters()).device)

            # Forward pass
            outputs = model(images)
            loss = F.cross_entropy(outputs, targets)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip)
            optimizer.step()

            # ✅ Step the scheduler *per batch* (OneCycleLR requirement)
            if scheduler:
                scheduler.step()

            # Track metrics
            train_losses.append(loss.detach())
            train_accs.append(accuracy(outputs, targets))

        # Validation
        model.eval()
        val_losses, val_accs = [], []
        with torch.no_grad():
            for images, targets in val_dl:
                images, targets = images.to(next(model.parameters()).device), targets.to(next(model.parameters()).device)
                outputs = model(images)
                loss = F.cross_entropy(outputs, targets)
                val_losses.append(loss)
                val_accs.append(accuracy(outputs, targets))

        # Compute averages
        train_loss_avg = torch.stack(train_losses).mean().item()
        train_acc_avg = sum(train_accs) / len(train_accs)
        val_loss_avg = torch.stack(val_losses).mean().item()
        val_acc_avg = sum(val_accs) / len(val_accs)
        lr = optimizer.param_groups[0]['lr']

        # Print epoch summary
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss_avg:.4f} | Train Acc: {train_acc_avg:.4f} | "
              f"Val Loss: {val_loss_avg:.4f} | Val Acc: {val_acc_avg:.4f} | LR: {lr:.5f}")

        # Save history
        history.append({
            "train_loss": train_loss_avg,
            "train_acc": train_acc_avg,
            "val_loss": val_loss_avg,
            "val_acc": val_acc_avg,
            "lr": lr
        })

    return history


# Hyperparameters

- **max_lr**: is the maximum learning rate that we set for learning rate scheduler. For the learning rate scheduler we used OneCycleLR, which sets the learning rate to a low learning rate, gradually increases it to the max learning rate then goes back to a low learning rate. <br/>
- **grad_clip**: prevents the gradients to become too large. <br/>
- **weight_decay**: essentially tries to make the model simple and helps the model generalise better.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.001,
    epochs=60,
    steps_per_epoch=len(train_dl),
    pct_start=0.45,
    div_factor=1)

# Training

In [ ]:
history = fit(
    model,
    train_dl,
    test_dl,
    optimizer,
    epochs=60,
    grad_clip=0.1 ,   # optional
    scheduler=scheduler  # optional
)


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")


# Plotting

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set general font style for high-impact journal
plt.rcParams.update({
    'font.size': 10,          # base font size
    'font.family': 'serif',   # or 'Times New Roman'
    'axes.labelsize': 10,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'lines.markersize': 4
})

def plot_accuracy(history, save_path="/kaggle/working/accuracycifar10.pdf"):
    epochs = range(1, len(history)+1)
    plt.figure(figsize=(3.5, 3))
    plt.plot(epochs, [x.get("train_acc") for x in history], 'o-', markersize=2, color='blue', label="Train Accuracy")
    plt.plot(epochs, [x.get("val_acc") for x in history], 'o-', markersize=2, color='red', label="Validation Accuracy")
    plt.xlabel("Epoch", fontsize=10, fontweight='bold')
    plt.ylabel("Accuracy", fontsize=10, fontweight='bold')
    plt.ylim(0.3, 1.0)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()


def plot_loss(history, save_path="/kaggle/working/losscifar10.pdf"):
    epochs = range(1, len(history)+1)
    plt.figure(figsize=(3.5, 3))
    plt.plot(epochs, [x.get("train_loss") for x in history], 'o-', markersize=2, color='blue', label="Train Loss")
    plt.plot(epochs, [x.get("val_loss") for x in history], 'o-', markersize=2, color='red', label="Validation Loss")
    plt.xlabel("Epoch", fontsize=10, fontweight='bold')
    plt.ylabel("Loss", fontsize=10, fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()


def plot_lrs(history, save_path="/kaggle/working/lrscifar10.pdf"):
    lrs = np.concatenate([x.get("lrs", []) for x in history])
    plt.figure(figsize=(3.5, 3))
    plt.plot(range(1, len(lrs)+1), lrs, '-', color='green', label="Learning Rate")
    plt.xlabel("Batch Number", fontsize=10, fontweight='bold')
    plt.ylabel("Learning Rate", fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()


In [ ]:
plot_loss(history)

In [ ]:
plot_accuracy(history)

In [ ]:
plot_lrs(history)

In [ ]:
import torch
import torch.nn as nn
from collections import OrderedDict

# -----------------------
# Pointwise Convolution (full-rank 1x1, mixes channels)
# -----------------------
class Pointwise_Conv(nn.Module):
    def __init__(self, in_fts, out_fts):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels=in_fts, 
                out_channels=out_fts, 
                kernel_size=1, 
                bias=False
            ),
            nn.BatchNorm2d(out_fts),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

# -----------------------
# Depthwise Convolution (per-channel spatial "blueprint" filter)
# -----------------------
class Depthwise_Conv(nn.Module):
    def __init__(self, fts, stride=(1, 1)):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels=fts, 
                out_channels=fts, 
                kernel_size=3, 
                stride=stride, 
                padding=1, 
                groups=fts, 
                bias=False
            ),
            nn.BatchNorm2d(fts),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

# -----------------------
# Blueprint Separable Conv block (BSConv-U, Haase & Amthor, CVPR 2020)
#
# This is the same building block used by InceptionBlockLight above: a
# full-rank 1x1 pointwise conv mixes channels FIRST (in_fts -> out_fts,
# producing "blueprint" feature maps), then a cheap depthwise conv applies
# one learned k x k spatial filter per output channel SECOND. That's the
# reverse order of classic depthwise-separable conv (depthwise first,
# pointwise second). Since the rest of the notebook has moved to BSConv and
# doesn't use plain depthwise-separable conv anywhere else, this baseline
# uses the same block here instead of mixing two different separable-conv
# designs across models.
# -----------------------
class BSConv_Block(nn.Module):
    def __init__(self, in_fts, out_fts, stride=(1, 1)):
        super().__init__()
        self.pw = Pointwise_Conv(in_fts, out_fts)
        self.dw = Depthwise_Conv(out_fts, stride)

    def forward(self, x):
        return self.dw(self.pw(x))

# -----------------------
# MobileNet (BSConv) for CIFAR-10
# -----------------------
class MyMobileNet_v1(nn.Module):
    def __init__(self, in_fts=3, num_filter=32, width_multiplier=1.0, num_classes=10):
        super().__init__()

        # Initial Conv layer
        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels=in_fts, 
                out_channels=int(width_multiplier * num_filter), 
                kernel_size=3, stride=2, padding=1, bias=False
            ),
            nn.BatchNorm2d(int(width_multiplier * num_filter)),
            nn.ReLU(inplace=True)
        )

        self.in_fts = int(width_multiplier * num_filter)

        # Layer configuration
        self.nlayer_filter = [
            width_multiplier * num_filter * 2,
            [width_multiplier * num_filter * pow(2, 2)],
            width_multiplier * num_filter * pow(2, 2),
            [width_multiplier * num_filter * pow(2, 3)],
            width_multiplier * num_filter * pow(2, 3),
            [width_multiplier * num_filter * pow(2, 4)],
            [5, width_multiplier * num_filter * pow(2, 4)],
            [width_multiplier * num_filter * pow(2, 5)],
            width_multiplier * num_filter * pow(2, 5)
        ]

        # BSConv blocks (replaces the plain depthwise-separable blocks)
        self.DSC = self.layer_construct()

        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d(1)

        # Fully connected layer (dynamic input size)
        self.fc = nn.Linear(int(self.in_fts), num_classes)

    def forward(self, x):
        N = x.size(0)
        x = self.conv(x)
        x = self.DSC(x)
        x = self.avgpool(x)
        x = x.view(N, -1)
        x = self.fc(x)
        return x

    def layer_construct(self):
        block = OrderedDict()
        index = 1
        for l in self.nlayer_filter:
            if isinstance(l, list):
                if len(l) == 2:  # repeat n times
                    for _ in range(l[0]):
                        block[str(index)] = BSConv_Block(self.in_fts, int(l[1]))
                        self.in_fts = int(l[1])
                        index += 1
                else:  # stride=2
                    block[str(index)] = BSConv_Block(self.in_fts, int(l[0]), stride=(2, 2))
                    self.in_fts = int(l[0])
                    index += 1
            else:
                block[str(index)] = BSConv_Block(self.in_fts, int(l))
                self.in_fts = int(l)
                index += 1
        return nn.Sequential(block)

# -----------------------
# Test
# -----------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    x = torch.randn(64, 3, 32, 32).to(device)

    # Initialize MobileNet (BSConv) for CIFAR-10
    mobilenet = MyMobileNet_v1(width_multiplier=0.25, num_classes=10).to(device)
    out = mobilenet(x)
    print("Output shape:", out.shape)  # Expected: torch.Size([64, 10])


In [ ]:

optimizer = torch.optim.Adam(mobilenet.parameters(), weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.001,
    epochs=60,
    steps_per_epoch=len(train_dl),
    pct_start=0.45,
    div_factor=1)
# fit for 60 epochs
mobilehistory = fit(
    mobilenet,
    train_dl,
    test_dl,
    optimizer,
    epochs=60,
    grad_clip=0.1,
    scheduler=scheduler# optional
)


In [ ]:

import torch
import torch.nn as nn
from torchvision import models 
# --- Model ---
class ResNet50_CIFAR10(BaseModel):
    def __init__(self, num_classes=10):
        super().__init__()
        self.model = models.resnet50(weights=None)   # <-- no pretrained weights

        # Modify first conv to take RGB (3 channels)
        self.model.conv1 = nn.Conv2d(
            3, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # Modify final fully connected layer for CIFAR-10 classes
        in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(in_features, num_classes)

    def forward(self, xb):
        return self.model(xb)


# --- Training ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = ResNet50_CIFAR10().to(device)

optimizer = torch.optim.Adam(resnet.parameters(), weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.001,
    epochs=60,
    steps_per_epoch=len(train_dl),
    pct_start=0.45,
    div_factor=1)
# Train for 60 epochs
resnethistory = fit(
    resnet,
    train_dl,
    test_dl,
    optimizer,
    epochs=60,
    grad_clip=0.1,   # optional
    scheduler=scheduler  # optional
)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# -----------------------
# Custom Inception V1 for CIFAR-10
# -----------------------
class InceptionV1_CIFAR10(nn.Module):
    def __init__(self, num_classes=10, in_channels=3):
        super().__init__()
        # Load torchvision GoogLeNet (Inception V1)
        self.model = models.googlenet(weights=None, aux_logits=False)  # Disable aux classifiers

        # Replace first conv layer for RGB CIFAR-10
        # Original: kernel_size=7, stride=2 → we use stride=1 for small CIFAR-10 images
        self.model.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=64,
            kernel_size=7,
            stride=1,        # preserve more spatial info
            padding=3,
            bias=False
        )

        # Replace the final classifier for CIFAR-10 (10 classes)
        in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

# -----------------------
# Test run
# -----------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Initialize model
    googlenet = InceptionV1_CIFAR10(num_classes=10, in_channels=3).to(device)

    # Dummy CIFAR-10 input: batch_size=4, 3 channels, 32x32
    x = torch.randn(4, 3, 32, 32).to(device)

    # Forward pass
    google = googlenet(x)
    print("Output shape:", google.shape)  # Expected: torch.Size([4, 10])


In [ ]:
optimizer = torch.optim.Adam(googlenet.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.001,
    epochs=60,
    steps_per_epoch=len(train_dl),
    pct_start=0.45,
    div_factor=1)
googlehistory = fit(
    googlenet,
    train_dl,
    test_dl,
    optimizer,
    epochs=60,
    grad_clip=0.1,   # optional
    scheduler=scheduler  # optional
)

In [ ]:
def plot_all_accuracies(histories, labels, max_epoch=25, save_path="/kaggle/working/comapreaccuracycifar10.pdf"):
    """
    Plot validation accuracy of multiple models and save the figure in Kaggle.

    histories : list of list of dict
        Each history is a list of dictionaries, e.g., [{"val_acc": 0.92}, ...]
    labels : list of str
        Labels for each history
    max_epoch : int
        Maximum number of epochs to plot
    save_path : str
        File path where the PDF will be saved
    """
    import matplotlib.pyplot as plt

    # Update plot style
    plt.rcParams.update({
        'font.size': 10,
        'font.family': 'serif',
        'axes.labelsize': 10,
        'axes.titlesize': 12,
        'legend.fontsize': 9,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'lines.markersize': 4
    })

    plt.figure(figsize=(4.5, 3.5))  # two-column style

    for history, label in zip(histories, labels):
        epochs = range(1, min(len(history), max_epoch)+1)
        val_accs = [x.get("val_acc") for x in history[:max_epoch]]
        plt.plot(epochs, val_accs, 'o-', markersize=2.5, label=label)

    plt.xlabel("Epoch", fontsize=10, fontweight='bold')
    plt.ylabel("Validation Accuracy", fontsize=10, fontweight='bold')
    plt.ylim(0.4, 1.0)
    plt.legend(loc='lower right', frameon=False)
    plt.tight_layout()
    
    # Save figure
    plt.savefig(save_path)
    print(f"Figure saved to {save_path}")
    
    # Show plot in notebook
    plt.show()


In [ ]:
histories = [googlehistory, resnethistory, mobilehistory, history]
labels = ["InceptionV1", "ResNet50", "MobileNet-BSConv", "SIB-Net"]

plot_all_accuracies(histories, labels, max_epoch=60)


In [ ]:
def plot_all_losses(histories, labels, max_epoch=15, save_path="/kaggle/working/comparelosscifar10.pdf"):
    """
    Plot validation loss of multiple models on CIFAR-10.
    """
    plt.rcParams.update({
        'font.size': 10,
        'font.family': 'serif',
        'axes.labelsize': 10,
        'axes.titlesize': 12,
        'legend.fontsize': 9,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'lines.markersize': 4
    })
    
    plt.figure(figsize=(4.5, 3.5))  # two-column style
    
    all_losses = []
    for history, label in zip(histories, labels):
        epochs = range(1, min(len(history), max_epoch)+1)
        val_losses = [x.get("val_loss") for x in history[:max_epoch]]
        all_losses.extend(val_losses)
        plt.plot(epochs, val_losses, 'o-', markersize=2.5, label=label)

    plt.xlabel("Epoch", fontsize=10, fontweight='bold')
    plt.ylabel("Validation Loss", fontsize=10, fontweight='bold')
    
    # y-axis from 0 to slightly above the max value (capped so early spikes don't flatten the curves)
    plt.ylim(0.0, min(max(all_losses) * 1.05, 2.0))
    
    plt.legend(loc='upper right', frameon=False)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

plot_all_losses(histories, labels, max_epoch=60, save_path="/kaggle/working/loss_all.pdf")


In [ ]:
def evaluate_test_set(models, labels, test_dl, device):
    """
    Evaluate multiple models on the test set and compute Accuracy, Recall, Precision, F1 Score.
    Returns a dictionary: {label: {metric: value}}
    """
    results = {}
    for model, label in zip(models, labels):
        model.eval()
        y_true = []
        y_pred = []

        with torch.no_grad():
            for X, y in test_dl:
                X, y = X.to(device), y.to(device)
                outputs = model(X)
                preds = outputs.argmax(dim=1)
                y_true.append(y.cpu().numpy())
                y_pred.append(preds.cpu().numpy())

        y_true = np.concatenate(y_true)
        y_pred = np.concatenate(y_pred)

        acc = np.mean(y_true == y_pred)
        rec = recall_score(y_true, y_pred, average="macro")
        prec = precision_score(y_true, y_pred, average="macro")
        f1 = f1_score(y_true, y_pred, average="macro")

        results[label] = {
            "Accuracy": acc,
            "Recall": rec,
            "Precision": prec,
            "F1_Score": f1
        }

    return results


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

def plot_model_comparison_bar(models, labels, test_dl, device, save_path="test_results.pdf"):
    """
    Plot grouped bar chart for Accuracy, Recall, Precision, F1 Score across multiple models.
    Metric values are displayed inside the bars, rotated 90°, near the top.
    Legend arranged in two columns.
    """
    results = evaluate_test_set(models, labels, test_dl, device)

    # Collect metrics
    metric_names = ["Accuracy", "Recall", "Precision", "F1_Score"]
    n_metrics = len(metric_names)
    n_models = len(labels)

    data = np.zeros((n_models, n_metrics))
    for i, label in enumerate(labels):
        for j, metric in enumerate(metric_names):
            data[i, j] = results[label][metric]

    # Bar positions
    x = np.arange(n_metrics)
    width = 0.8 / n_models

    # Warm color palette
    colors = ["#3b0a45", "#9b1c31", "#e66101", "#fdb863"]

    plt.rcParams.update({
        "font.size": 10,
        "font.family": "serif"
    })

    fig, ax = plt.subplots(figsize=(5.2, 4.5))

    for i, (label, color) in enumerate(zip(labels, colors)):
        offset = (i - (n_models - 1) / 2) * width
        bars = ax.bar(
            x + offset, data[i], width,
            label=label, color=color,
            alpha=0.7, edgecolor="black", linewidth=0.5
        )

        # Add rotated labels inside near the top
        for bar, val in zip(bars, data[i]):
            bar_color = np.array(bar.get_facecolor()[:3])
            brightness = np.dot(bar_color, [0.299, 0.587, 0.114])
            text_color = "white" if brightness < 0.5 else "black"

            y_pos = min(bar.get_height(), 1.0) - 0.005
            
            ax.text(
                bar.get_x() + bar.get_width()/ 1.5,
                y_pos,   # slightly lower so it never leaves bar
                f"{val:.4f}",
                ha="center", va="top",
                rotation=90,
                fontsize=9,
                fontweight="bold",
                color=text_color
            )

    # Axes & labels
    ax.set_xticks(x)
    ax.set_xticklabels(
        metric_names,
        fontweight="bold",
        bbox=dict(facecolor="#a6d96a", edgecolor="green", boxstyle="round,pad=0.2")
    )
    ax.set_ylabel("Value", fontweight="bold")

    # Slightly raise y-limit so text fits perfectly
    ax.set_ylim(0.7, 1.0)


    # Two-column legend
    ax.legend(
        loc="upper center",
        frameon=True,
        ncol=2,
        bbox_to_anchor=(0.5, 0.99),
        fontsize=8,
        columnspacing=0.8,
        handletextpad=0.5
    )

    # Style tweaks
    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)
    ax.yaxis.grid(False)

    plt.subplots_adjust(top=0.80)
    plt.savefig(save_path, dpi=592, bbox_inches="tight")
    plt.show()

    return results
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models_list = [mobilenet, model, resnet, googlenet]
labels = ["MobilenetV1", "SIB-Net", "Resnet50", "InceptionV1"]

results = plot_model_comparison_bar(models_list, labels, test_dl, device)
print(results)

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def get_correct_class_probabilities(model, test_dl, device):
    """
    Returns the predicted probabilities for the true class (correct class) for each sample.
    """
    model.eval()
    model.to(device)
    probs = []

    with torch.no_grad():
        for images, labels in test_dl:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            softmax_probs = torch.nn.functional.softmax(outputs, dim=1)
            # probability of the correct class
            correct_probs = softmax_probs.gather(1, labels.view(-1, 1)).squeeze()
            probs.extend(correct_probs.cpu().numpy())

    return np.array(probs)


def plot_detection_density(models, labels, test_dl, device, save_path="detection_density_cifar10.pdf"):
    plt.figure(figsize=(3.5, 3))

    # custom colors to match Fig. 4
    colors = {
        "SIB-Net": "green",
        "ResNet50": "blue",
        "InceptionV1": "purple",
        "MobileNet-BSConv": "orange"
    }

    for model, label in zip(models, labels):
        probs = get_correct_class_probabilities(model, test_dl, device)
        
        # histogram for MobileNet
        if "MobileNet" in label:
            plt.hist(probs, bins=30, density=True, alpha=0.6, color=colors[label])
        
        # KDE line for all models
        sns.kdeplot(probs, label=label, color=colors[label], linewidth=1.5, clip=(0, 1))

    plt.xlabel("Probability", fontweight="bold")
    plt.ylabel("Density", fontweight="bold")

    # 🔧 Fixed title box (no more visible orange rectangle)
    plt.title(
        "CIFAR-10",
        fontsize=12,
        fontweight="bold",
        pad=12,
        bbox=dict(facecolor="white", edgecolor="gray", alpha=0.5)  # soft translucent box
    )

    plt.xlim(0, 1)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()


# ===== Usage Example =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Replace these placeholders with your actual model objects and test dataloader
# e.g. googlenet, resnet, mobilenet, sibnet_model
models_list = [googlenet, resnet, mobilenet, model]
labels = ["InceptionV1", "ResNet50", "MobileNet-BSConv", "SIB-Net"]

plot_detection_density(models_list, labels, test_dl, device)


# Make predictions and see the image with its result


In [ ]:
def predict_image(img, model):
    xb = to_device(img.unsqueeze(0), device)
    yb = model(xb)
    _, preds  = torch.max(yb, dim=1)
    return test_data.classes[preds[0].item()]

In [ ]:
img, label = test_data[0]
plt.imshow(denormalize(img).permute(1, 2, 0))  # un-normalize, CHW -> HWC
print('Label:', test_data.classes[label], ', Predicted:', predict_image(img, model))

In [ ]:
img, label = test_data[1002]
plt.imshow(denormalize(img).permute(1, 2, 0))  # un-normalize, CHW -> HWC
print('Label:', test_data.classes[label], ', Predicted:', predict_image(img, model))

# Save

In [ ]:
torch.save(model.state_dict(), 'cifar10-sibnet-project.pth')

In [ ]:
import torch

def count_parameters(model):
    """Return the total number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ===== Example Usage =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Replace with your actual model objects
models_list = [mobilenet, googlenet, model, resnet]
labels = ["MobileNet-BSConv", "InceptionV1", "SIB-Net", "Resnet50"]

# Print parameter counts
print("Trainable parameters per model:\n")
for label, mdl in zip(labels, models_list):
    params = count_parameters(mdl)
    print(f"{label:<12} : {params:,} parameters ({params/1e6:.2f}M)")
